# Connect and Show

This notebook is the **hello world** for `mitk-workbench-remote`. You will learn how to:

- Connect to a running MITK Workbench
- Inspect the connection with `wb.ping()` and `wb.info`
- Show a numpy array in the Workbench with `wb.show()`
- Set visual properties (name, color, opacity)
- Browse the DataStorage
- Enable debug logging

**Prerequisites:** A running MITK Workbench instance with the REST API enabled.

## 1. Connect to a running Workbench

`mw.connect()` creates a `Workbench` handle. No HTTP call is made until you use it —
the connection is established lazily on first access.

Use `wb.ping()` to verify reachability, and `wb.info` to inspect the server.

In [14]:
import mitk_workbench_remote as mw

wb = mw.connect("http://localhost:8080")
print(f"Reachable: {wb.ping()}")
print(f"Server:    {wb.info.name}")
print(f"MITK:      {wb.info.mitk_version}")
print(f"API:       {wb.info.api_version}")

Reachable: True
Server:    MITK Workbench REST API
MITK:      2025.12.99-23b4e629
API:       v1


## 2. Show a numpy array

`wb.show()` is the primary one-liner for displaying data.  It creates a node,
uploads the data, sets display properties, and reinitialises the render views —
all in a single call.

The return value is the newly created `DataNode`.

In [15]:
import numpy as np

# Create a simple gradient volume
size = 64
arr = np.sum(np.indices((size, size, size), dtype=np.float32), axis=0)
arr /= arr.max()

node = wb.show(arr, name="Gradient")
print(f"Created node: {node}")

Created node: <DataNode 'Gradient' (Image) @ /Gradient uid=node_3>


## 3. Visual properties

`show()` accepts `color` (RGB tuple, 0.0-1.0) and `opacity` (0.0-1.0) as keyword
arguments.

In [16]:
# Show a second volume with visual properties set at creation
mask = (arr > 0.5).astype(np.uint8)
mask_node = wb.show(mask, name="Threshold", color=(1.0, 0.0, 0.0), opacity=0.5)

You can also change them afterwards via the node's properties. The new property values will be directly transferred. In order to update the rendering, you have to call update on the workbench.

In [13]:
# Change properties after creation
mask_node.color = (0.0, 1.0, 0.0)
mask_node.opacity = 0.2
wb.update()

print(f"Color:   {mask_node.color}")
print(f"Opacity: {mask_node.opacity}")

Color:   (0.0, 1.0, 0.0)
Opacity: 0.20000000298023224


## 4. Browse the DataStorage

The `wb.storage` object provides iteration, `len()`, and `__getitem__` access
over all nodes in the Workbench's DataStorage.  In Jupyter, it renders as an
HTML table.

In [8]:
print(f"Total nodes: {len(wb.storage)}\n")

for n in wb.storage:
    print(f"  {n.uid:10s}  {n._name:20s}  type={n.data_type}")

Total nodes: 6

  node_3      stdmulti.widget0.plane  type=PlaneGeometryData
  node_5      stdmulti.widget2.plane  type=PlaneGeometryData
  node_6      stdmulti.widget1.plane  type=PlaneGeometryData
  node_4      Widgets               type=None
  node_2      Threshold             type=Image
  node_1      Gradient              type=Image


In [9]:
# Rich display in Jupyter
wb.storage

uid,name,type,path
node_3,stdmulti.widget0.plane,PlaneGeometryData,/Widgets/stdmulti.widget0.plane
node_5,stdmulti.widget2.plane,PlaneGeometryData,/Widgets/stdmulti.widget2.plane
node_6,stdmulti.widget1.plane,PlaneGeometryData,/Widgets/stdmulti.widget1.plane
node_4,Widgets,,/Widgets
node_2,Threshold,Image,/Threshold
node_1,Gradient,Image,/Gradient


## 5. Enable debug logging

`mitk-workbench-remote` uses Python's standard `logging` module.  By default
all output is suppressed (NullHandler).  Enable it to see HTTP requests,
transfer mode negotiation, and other internals.

You can also filter to a specific sub-logger, e.g.
`logging.getLogger("mitk_workbench_remote.transport").setLevel(logging.DEBUG)`.

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG, format="%(name)s %(levelname)s %(message)s")

# This call will now print DEBUG output showing the HTTP request
wb.ping()

In [ ]:
# Reset to suppress logging for the rest of the notebook
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("mitk_workbench_remote").setLevel(logging.WARNING)

## 6. Clean up

In [ ]:
node.remove()
mask_node.remove()
print("Done.")